# All-monkey presentation figures

Presentation figures in the `OC_presentation.ipynb` style (same colours, font sizes and line widths), computed for every monkey listed in `cfg.monkeys`.

- **Fig 1**: response magnitude and latency of the static conditions vs the movie.
- **Fig 2**: static–dynamic representational similarity (dRSA, permutation significance, split-half reliability, timed frames, earlier-frame regression).
- **Fig 3**: model-layer RSA timecourses and layer depth vs latency (temporal score).

The most recent sessions and reliable-channel lists of every monkey are set in `MONKEY_SESSIONS` at the top of the configuration cell.

In [ ]:
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import yaml
from scipy.stats import spearmanr, ttest_rel

# Locate the repository whether the notebook starts in the project root or scripts/.
working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (working_directory, *working_directory.parents)
    if (candidate / 'config.yaml').is_file()
)
ENV = os.getenv('MY_ENV', 'tiziano_mac_mini')
with open(PROJECT_ROOT / 'config.yaml', 'r') as file:
    config = yaml.safe_load(file)
# end with open

paths = config[ENV]['paths']
for source_path in (paths['src_path'], paths['useful_stuff_path']):
    if source_path not in sys.path:
        sys.path.append(source_path)
    # end if source_path
# end for source_path

from project_specific_utils import (
    compute_rdm_timeseries, compute_split_half_reliability,
    depth_latency_score, load_natraster, match_timed_static_movie_rasters,
    model_frame_similarity_by_layer, permutation_significance_mask,
    plot_significance_masked_matrix, regressed_static_dynamic_drsa,
    rowwise_similarity, summarize_layer_timing, window_mean_responses,
    window_smoothed_response_latencies,
)
from project_specific_utils.last_frame_decay import resample_array
from project_specific_utils.timed_frame_window_rsa import (
    compute_timed_frame_window_rsa, load_dynamic_presentations,
    sliding_movie_window_rsa,
)
from useful_stuff.general_utils import truncate_colormap

## Configuration

`MONKEY_SESSIONS` holds, for every monkey, the most recent averaged image (`*_natraster_img.mat`) and movie (`*_natraster_vid.mat`) sessions, the presentation-level movie raster (`*_raster_vid.mat`, used only for the split-half reliability of Fig 2b), the one-based inclusive MATLAB channel range, and the key of its list in `reliable_channels.yaml`. Select the monkeys to plot with `cfg.monkeys`.

Windows are half-open `(start_ms, end_ms)`: static windows are measured from image onset, dynamic windows from movie onset. Pooled figures (1b, 1d, 1f) colour every channel by monkey with `MONKEY_COLORS`.

In [ ]:
# Most recent sessions of every monkey. The presentation-level movie raster
# can lag behind the averaged natraster: it is only used for Fig 2b.
MONKEY_SESSIONS = {
    'baby1': {
        'dynamic_exp_name': 'baby1_260716to24',
        'static_exp_name': 'baby1_260718to27',
        'dynamic_raster_exp_name': 'baby1_260716to24',
        'good_channels': (84, 186),
        'reliable_channels_key': 'baby1_260718to27',
    },
    'paul': {
        'dynamic_exp_name': 'paul_20260831to0927',
        'static_exp_name': 'paul_20260901to0925',
        'dynamic_raster_exp_name': 'paul_20260831to0919',
        'good_channels': (1, 64),
        'reliable_channels_key': 'paul_20260901to0923',
    },
    'red': {
        'dynamic_exp_name': 'red_20260720to0927',
        'static_exp_name': 'red_20260726to0925',
        'dynamic_raster_exp_name': 'red_20260720to0921',
        'good_channels': (1, 64),
        'reliable_channels_key': 'red_20260726to0923',
    },
}
# Same monkey colours as run_last_frame_residual_hierarchy.py.
MONKEY_COLORS = {'baby1': '#3474BD', 'red': '#C4564C', 'paul': '#11A579'}


@dataclass
class Cfg:
    # Any subset of MONKEY_SESSIONS, in plotting order.
    monkeys: tuple[str, ...] = ('baby1', 'paul', 'red')
    use_reliable_channels: bool = True
    source_fs: float = 1000
    new_fs: float = 100
    static_crop_ms: float = 1000
    # Neural RDMs and RDM comparisons (match the saved permutation results).
    neural_rdm_metric: str = 'cosine_cnt'
    rsa_metric: str = 'spearman'
    # Movie time at which every static condition appears.
    image_2000ms_onset_ms: float = 2000
    image_2250ms_onset_ms: float = 2250
    last_frame_onset_ms: float = 2500
    video_plot_end_ms: float = 3500

    # Fig 1a: average activity.
    average_activity_figsize: tuple[float, float] = (11, 5)
    average_activity_error_alpha: float = 0.2
    # Fig 1b: paired t of the window-mean activity (static - dynamic).
    # Static condition: 'last_frame', '2000ms' or '2250ms'.
    mag_difference_static_response: str = 'last_frame'
    mag_difference_static_window_ms: tuple[float, float] = (100, 500)
    mag_difference_dynamic_window_ms: tuple[float, float] = (2500, 2900)
    # Fig 1c-1d: window-mean responses correlated across stimuli.
    tuning_static_window_ms: tuple[float, float] = (100, 500)
    tuning_dynamic_window_ms: tuple[float, float] = (2500, 2900)
    tuning_scatter_figsize: tuple[float, float] = (7, 7)
    tuning_scatter_size: float = 180
    channel_bar_figsize: tuple[float, float] = (16, 5)
    channel_correlation_ylim: tuple[float, float] = (-1, 1)
    channel_tick_size: float = 9
    # Fig 1e-1f: Gaussian-smoothed peak latencies (last frame vs movie).
    latency_static_window_ms: tuple[float, float] = (80, 250)
    latency_dynamic_window_ms: tuple[float, float] = (2580, 2750)
    latency_smoothing_sigma_ms: float = 20
    # Pooled channel histograms (1b, 1f) and the population histogram (1e).
    histogram_bin_count: int = 15
    histogram_figsize: tuple[float, float] = (8, 5)
    histogram_stack_by_monkey: bool = True

    # Fig 2a: saved permutation results (static_dynamic_drsa_permutation.ipynb).
    # Condition: 'raw' or '2250ms-signal-regressed'.
    permutation_condition: str = 'raw'
    n_permutations: int = 1000
    permutation_seed: int = 0
    # Correction: 'pointwise' (uncorrected), 'max_statistic' (FWE) or 'cluster'.
    significance_correction: str = 'cluster'
    significance_alpha: float = 0.05
    nonsignificant_blur_sigma_bins: float = 2
    nonsignificant_fade_alpha: float = 0.5
    significance_contour_color: str = 'white'
    significance_contour_width: float = 2
    static_dynamic_matrix_figsize: tuple[float, float] = (10, 4)
    static_dynamic_matrix_cmap: str = 'viridis'
    # None lets every monkey use its own limit.
    static_dynamic_matrix_vmin: float | None = 0
    static_dynamic_matrix_vmax: float | None = None
    # Fig 2b: best static match per movie time vs split-half RDM reliability.
    split_half_static_reference_window_ms: tuple[float, float] = (0, 1000)
    split_half_n_repeats: int = 20
    split_half_seed: int = 0
    split_half_use_spearman_brown: bool = True
    split_half_spearman_brown_k: float = 2
    split_half_figsize: tuple[float, float] = (10, 4)
    split_half_reliability_color: str = 'black'
    split_half_static_dynamic_color: str = 'tab:orange'
    split_half_std_alpha: float = 0.2
    # Fig 2c: static frame RDMs vs sliding movie window (timed_frame_window_rsa).
    window_rsa_static_window_ms: tuple[float, float] = (80, 250)
    window_rsa_movie_window_ms: tuple[float, float] = (2580, 2750)
    # Centering happens after window averaging, so cosine equals cosine_cnt.
    window_rsa_rdm_metric: str = 'cosine'
    window_rsa_similarity_metric: str = 'spearman'
    window_rsa_step_ms: float = 10
    window_rsa_show_target_window: bool = True
    window_rsa_figsize: tuple[float, float] = (11, 5)
    # Fig 2d: earlier static responses regressed out of the last frame.
    regressed_conditions: tuple[str, ...] = ('2000ms', '2250ms')
    max_similarity_colors: dict = field(default_factory=lambda: {
        'raw': 'tab:orange', '2000ms': '#7596C5', '2250ms': '#4A78B5',
    })

    # Fig 3: model-layer RSA.
    model_name: str = 'ijepa_vith14_1k'
    model_dataset_name: str = 'static_dynamic'
    model_pooling: str | None = 'mean'
    model_features_dir: Path | None = None
    model_rdm_metric: str = 'cosine_cnt'
    first_frame_ms: float = 0
    previous_frame_ms: float = 2250
    residual_fit_intercept: bool = True
    smoothing: float = 3  # Gaussian sigma in samples at new_fs.
    # One latency window per condition, shared by peaks and centroids; the
    # last frame and its residual always share last_frame_latency_window_ms.
    static_latency_window_ms: tuple[float, float] = (0, 300)
    first_frame_latency_window_ms: tuple[float, float] = (0, 300)
    last_frame_latency_window_ms: tuple[float, float] = (2300, 2700)
    centroid_similarity_cutoff: float = 0.01
    peak_similarity_cutoff: float = 0.01
    layer_timecourse_figsize: tuple[float, float] = (11, 5)
    # None fits the y axis to every monkey's curves.
    layer_timecourse_ylim: tuple[float, float] | None = None
    # Displayed time range per condition; None shows the whole response.
    layer_timecourse_xlims_ms: dict = field(default_factory=lambda: {
        'static': (0, 1000), 'first_frame': None, 'last_frame': None,
        'last_frame_residual': None,
    })
    show_latency_window: bool = True
    latency_line_centroid_tick_height: float = 0.05
    latency_line_centroid_tick_width: float = 3
    latency_scatter_figsize: tuple[float, float] = (8, 5)
    latency_scatter_cmap: str = 'plasma'
    latency_scatter_cmap_min: float = 0.1
    latency_scatter_cmap_max: float = 0.9
    latency_scatter_size: float = 200
    latency_scatter_edge_color: str = 'black'
    latency_scatter_edge_width: float = 0.8
    latency_scatter_n_ticks: int = 6
    latency_scatter_x_padding_fraction: float = 0.08
    latency_scatter_min_x_padding_ms: float = 10
    latency_scatter_y_ticks: tuple[float, ...] = (0, .2, .4, .6, .8, 1)
    latency_scatter_stats_fontsize: int = 12

    # OC_presentation style.
    linewidth: float = 3
    ticks_size: float = 20
    label_size: float = 25
    legend_size: float = 15
    title_size: float = 20
    show_monkey_titles: bool = True
    frame_500_color: str = '#7596C5'
    frame_250_color: str = '#4A78B5'
    static_color: str = '#3474BD'
    dynamic_color: str = '#7F3C8D'
    fig_save: bool = False
    figs_dir: str = f"{paths['data_path']}/results/figs_presentation_all_monkeys"
# EOC


cfg = Cfg()
unknown_monkeys = set(cfg.monkeys) - set(MONKEY_SESSIONS)
if unknown_monkeys:
    raise KeyError(f'Monkeys missing from MONKEY_SESSIONS: {sorted(unknown_monkeys)}')
# end if unknown monkeys
if cfg.mag_difference_static_response not in ('last_frame', '2000ms', '2250ms'):
    raise ValueError("mag_difference_static_response must be 'last_frame', '2000ms' or '2250ms'.")
# end if invalid static response
if cfg.significance_correction not in ('pointwise', 'max_statistic', 'cluster'):
    raise ValueError("significance_correction must be 'pointwise', 'max_statistic' or 'cluster'.")
# end if invalid correction
if not set(cfg.regressed_conditions) <= {'2000ms', '2250ms'}:
    raise ValueError("regressed_conditions may contain only '2000ms' and '2250ms'.")
# end if invalid regressed condition
if cfg.smoothing <= 0:
    raise ValueError('smoothing must be positive.')
# end if cfg.smoothing
if not 0 <= cfg.latency_scatter_cmap_min < cfg.latency_scatter_cmap_max <= 1:
    raise ValueError('The latency colormap limits must satisfy 0 <= min < max <= 1.')
# end if invalid colormap limits
for window_name in (
        'static_latency_window_ms', 'first_frame_latency_window_ms',
        'last_frame_latency_window_ms', 'mag_difference_static_window_ms',
        'mag_difference_dynamic_window_ms', 'latency_static_window_ms',
        'latency_dynamic_window_ms',
        ):
    window_start_ms, window_end_ms = getattr(cfg, window_name)
    if window_start_ms >= window_end_ms:
        raise ValueError(f'{window_name} must be increasing.')
    # end if window is not increasing
# end for window_name
# Frame onsets, used to express movie latencies relative to each frame.
STATIC_ONSETS_MS = {
    '2000ms': cfg.image_2000ms_onset_ms,
    '2250ms': cfg.image_2250ms_onset_ms,
    'last_frame': cfg.last_frame_onset_ms,
}


"""
style_axis
Apply the OC_presentation font sizes and spines to one axis.
Script-specific: it reads the notebook's cfg.

INPUT:
    - axis: matplotlib.axes.Axes -> axis to style
    - xlabel: str -> x-axis label
    - ylabel: str -> y-axis label
    - title: str | None -> optional title, shown when cfg.show_monkey_titles
    - hide_top_right: bool -> hide the top and right spines

OUTPUT:
    - None
"""
def style_axis(axis, xlabel, ylabel, title=None, hide_top_right=True):
    axis.set_xlabel(xlabel, fontsize=cfg.label_size)
    axis.set_ylabel(ylabel, fontsize=cfg.label_size)
    axis.tick_params(axis='both', labelsize=cfg.ticks_size)
    axis.spines['top'].set_visible(not hide_top_right)
    axis.spines['right'].set_visible(not hide_top_right)
    if title is not None and cfg.show_monkey_titles:
        axis.set_title(title, fontsize=cfg.title_size)
    # end if title shown
# EOF


"""
save_figure
Save one figure as PDF in cfg.figs_dir when cfg.fig_save is set.

INPUT:
    - figure: matplotlib.figure.Figure -> figure to save
    - file_stem: str -> file name without extension

OUTPUT:
    - None
"""
def save_figure(figure, file_stem):
    if cfg.fig_save:
        Path(cfg.figs_dir).mkdir(parents=True, exist_ok=True)
        figure.savefig(f'{cfg.figs_dir}/{file_stem}.pdf', bbox_inches='tight')
    # end if cfg.fig_save
# EOF


cfg

## Load and align every monkey

The same physical channels are kept in the image and movie sessions, and only stimuli shown in all four conditions (last frame, 2000-ms frame, 2250-ms frame, movie) are retained, in one shared order. Rasters are kept at `source_fs` (channels × time × stimuli) and also resampled to `new_fs`.

In [ ]:
data_dir = Path(paths['data_path']) / 'data'
reliable_channels_config = (
    PROJECT_ROOT / 'reliable_channels.yaml' if cfg.use_reliable_channels else None
)
static_crop_samples = int(round(cfg.static_crop_ms * cfg.source_fs / 1000))

monkey_data = {}
for monkey in cfg.monkeys:
    sessions = MONKEY_SESSIONS[monkey]
    channel_kwargs = {
        'good_channels': sessions['good_channels'],
        'reliable_channels_config': reliable_channels_config,
        'reliable_channels_key': sessions['reliable_channels_key'],
    }
    static_rasters, static_names, static_channels = load_natraster(
        data_dir / f"{sessions['static_exp_name']}_natraster_img.mat",
        return_channel_numbers=True, **channel_kwargs,
    )
    dynamic_rasters, dynamic_names, dynamic_channels = load_natraster(
        data_dir / f"{sessions['dynamic_exp_name']}_natraster_vid.mat",
        return_channel_numbers=True, **channel_kwargs,
    )
    if not np.array_equal(static_channels, dynamic_channels):
        raise ValueError(f'{monkey}: sessions retained different channels.')
    # end if channel numbers differ

    (
        last_frame_rasters, video_rasters, image_2000ms_rasters,
        image_2250ms_rasters, shared_stimuli, aligned_names,
    ) = match_timed_static_movie_rasters(
        static_rasters, static_names, dynamic_rasters, dynamic_names,
    )
    # channels x time x stimuli at source_fs; static responses cropped.
    rasters = {
        'last_frame': last_frame_rasters[:, :static_crop_samples, :],
        '2000ms': image_2000ms_rasters[:, :static_crop_samples, :],
        '2250ms': image_2250ms_rasters[:, :static_crop_samples, :],
        'dynamic': video_rasters,
    }
    monkey_data[monkey] = {
        'channel_numbers': static_channels,
        'shared_stimuli': shared_stimuli,
        'aligned_names': aligned_names,
        'rasters': rasters,
        'resampled': {
            condition: resample_array(condition_rasters, cfg.source_fs, cfg.new_fs)
            for condition, condition_rasters in rasters.items()
        },
    }
    print(
        f"{monkey}: {sessions['static_exp_name']} (img) + "
        f"{sessions['dynamic_exp_name']} (vid) | {len(static_channels)} channels | "
        f"{len(shared_stimuli)} stimuli"
    )
# end for monkey

## Fig 1a — Average response to the static conditions and to the movie

Channels are averaged first; lines are the mean across matched stimuli and shaded bands the SEM. Every static condition is drawn from the movie time at which its frame appears (2000 ms = −500 ms, 2250 ms = −250 ms, last frame = 2500 ms). The y axis is fitted to each monkey's responses.

In [ ]:
activity_conditions = (
    ('dynamic', 0, 'dynamic condition', cfg.dynamic_color),
    ('2000ms', cfg.image_2000ms_onset_ms, 'static condition (-500 ms)', cfg.frame_500_color),
    ('2250ms', cfg.image_2250ms_onset_ms, 'static condition (-250 ms)', cfg.frame_250_color),
    ('last_frame', cfg.last_frame_onset_ms, 'static condition (last frame)', cfg.static_color),
)
for monkey in cfg.monkeys:
    resampled = monkey_data[monkey]['resampled']
    figure, axis = plt.subplots(figsize=cfg.average_activity_figsize)
    for condition, onset_ms, label, color in activity_conditions:
        # time x stimuli after averaging channels.
        activity_by_stimulus = resampled[condition].mean(axis=0)
        activity = activity_by_stimulus.mean(axis=1)
        activity_sem = (
            activity_by_stimulus.std(axis=1, ddof=1)
            / np.sqrt(activity_by_stimulus.shape[1])
        )
        time_ms = onset_ms + np.arange(activity.size) * 1000 / cfg.new_fs
        plot_mask = time_ms < cfg.video_plot_end_ms
        axis.fill_between(
            time_ms[plot_mask], (activity - activity_sem)[plot_mask],
            (activity + activity_sem)[plot_mask], color=color,
            alpha=cfg.average_activity_error_alpha, label='_nolegend_', zorder=1,
        )
        axis.plot(
            time_ms[plot_mask], activity[plot_mask], color=color, label=label,
            linewidth=cfg.linewidth + 2, zorder=2,
        )
        if onset_ms > 0:
            axis.axvline(onset_ms, color=color, linestyle='--', linewidth=1, alpha=0.5)
        # end if static onset
    # end for condition
    axis.set_xlim(0, cfg.video_plot_end_ms)
    axis.legend(fontsize=cfg.legend_size)
    style_axis(axis, 'Time from video onset (ms)', r'Avg neural activity ($\mu$V)', monkey)
    figure.tight_layout()
    save_figure(figure, f'{monkey}_fig1a_average_activity')
    plt.show()
# end for monkey

## Fig 1b — Static − dynamic activity per channel (paired t score, pooled across monkeys)

For every channel, the response is averaged in `mag_difference_static_window_ms` (static condition `mag_difference_static_response`) and in `mag_difference_dynamic_window_ms` (movie). The paired t score across matched stimuli (`ttest_rel(static, dynamic)`) is positive when the static response is larger. Channels of all monkeys are pooled in one histogram (stacked by monkey when `histogram_stack_by_monkey`); the dashed line is the pooled mean.

In [ ]:
"""
plot_pooled_channel_histogram
Histogram of one value per channel, pooled over monkeys, optionally stacked
by monkey. Script-specific: it reads the notebook's cfg and colours.

INPUT:
    - values_by_monkey: dict[str, np.ndarray] -> one value per channel
    - xlabel: str -> x-axis label
    - ylabel: str -> y-axis label

OUTPUT:
    - figure: matplotlib.figure.Figure -> histogram figure
    - pooled_values: np.ndarray -> finite pooled values
"""
def plot_pooled_channel_histogram(values_by_monkey, xlabel, ylabel='Number of channels'):
    finite_values = {
        monkey: values[np.isfinite(values)]
        for monkey, values in values_by_monkey.items()
    }
    pooled_values = np.concatenate(list(finite_values.values()))
    bin_edges = np.histogram_bin_edges(pooled_values, bins=cfg.histogram_bin_count)
    figure, axis = plt.subplots(figsize=cfg.histogram_figsize)
    if cfg.histogram_stack_by_monkey:
        axis.hist(
            list(finite_values.values()), bins=bin_edges, stacked=True,
            color=[MONKEY_COLORS[monkey] for monkey in finite_values],
            label=list(finite_values), edgecolor='white',
        )
        axis.legend(frameon=False, fontsize=cfg.legend_size)
    else:
        axis.hist(pooled_values, bins=bin_edges, color=cfg.dynamic_color, edgecolor='white')
    # end if stacked by monkey
    axis.axvline(0, color='0.25', linewidth=1)
    axis.axvline(pooled_values.mean(), color='black', linestyle='--', linewidth=1.5)
    style_axis(axis, xlabel, ylabel)
    figure.tight_layout()
    return figure, pooled_values
# EOF


magnitude_t_scores = {}
for monkey in cfg.monkeys:
    rasters = monkey_data[monkey]['rasters']
    # channels x stimuli window means.
    static_responses, _ = window_mean_responses(
        rasters[cfg.mag_difference_static_response],
        cfg.mag_difference_static_window_ms, cfg.source_fs,
    )
    dynamic_responses, _ = window_mean_responses(
        rasters['dynamic'], cfg.mag_difference_dynamic_window_ms, cfg.source_fs,
    )
    # One paired t per channel, with stimuli as the paired observations.
    magnitude_t_scores[monkey] = ttest_rel(
        static_responses, dynamic_responses, axis=1,
    ).statistic
# end for monkey

figure, pooled_t_scores = plot_pooled_channel_histogram(
    magnitude_t_scores, 'Paired t score (static − dynamic)',
)
save_figure(figure, 'all_monkeys_fig1b_activity_difference_tscores')
plt.show()
print(
    f'{pooled_t_scores.size} channels | mean t = {pooled_t_scores.mean():.2f} | '
    f'static > dynamic in {np.sum(pooled_t_scores > 0)}, '
    f'dynamic > static in {np.sum(pooled_t_scores < 0)}'
)

## Fig 1c — Population-average static vs dynamic response

One point per matched stimulus: the last-frame response averaged in `tuning_static_window_ms` against the movie response averaged in `tuning_dynamic_window_ms`, both averaged over channels. The dashed line marks equal responses.

In [ ]:
tuning_responses = {}
for monkey in cfg.monkeys:
    rasters = monkey_data[monkey]['rasters']
    static_channel_responses, _ = window_mean_responses(
        rasters['last_frame'], cfg.tuning_static_window_ms, cfg.source_fs,
    )
    dynamic_channel_responses, _ = window_mean_responses(
        rasters['dynamic'], cfg.tuning_dynamic_window_ms, cfg.source_fs,
    )
    # channels x stimuli, reused for the per-channel correlations of Fig 1d.
    tuning_responses[monkey] = (static_channel_responses, dynamic_channel_responses)

    static_population_responses = static_channel_responses.mean(axis=0)
    dynamic_population_responses = dynamic_channel_responses.mean(axis=0)
    population_rank_correlation = spearmanr(
        static_population_responses, dynamic_population_responses,
    ).statistic

    # Give both axes the same limits so the dashed line marks equal responses.
    activity_min = min(static_population_responses.min(), dynamic_population_responses.min())
    activity_max = max(static_population_responses.max(), dynamic_population_responses.max())
    activity_padding = 0.05 * max(activity_max - activity_min, 1e-12)
    activity_limits = (activity_min - activity_padding, activity_max + activity_padding)

    figure, axis = plt.subplots(figsize=cfg.tuning_scatter_figsize)
    axis.scatter(
        static_population_responses, dynamic_population_responses,
        color=cfg.dynamic_color, s=cfg.tuning_scatter_size, alpha=0.8,
        edgecolor='white', linewidth=1, zorder=3,
    )
    axis.plot(
        activity_limits, activity_limits, '--', color='0.35',
        linewidth=cfg.linewidth, label='Identity', zorder=2,
    )
    axis.set(xlim=activity_limits, ylim=activity_limits, aspect='equal')
    style_axis(
        axis, r'static population response ($\mu$V)',
        r'dynamic population response ($\mu$V)', monkey,
    )
    axis.legend(frameon=False, loc='lower right', fontsize=cfg.legend_size)
    axis.text(
        0.05, 0.95, fr'Spearman $\rho$ = {population_rank_correlation:.3f}',
        transform=axis.transAxes, ha='left', va='top', fontsize=cfg.ticks_size,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor='black', alpha=0.9),
        zorder=4,
    )
    figure.tight_layout()
    save_figure(figure, f'{monkey}_fig1c_population_static_dynamic_responses')
    plt.show()
# end for monkey

## Fig 1d — Static–dynamic correlation per channel

Spearman ρ across matched stimuli between each channel's static and dynamic window means (same windows as Fig 1c). Channels are grouped and coloured by monkey and labelled with their MATLAB channel number.

In [ ]:
channel_correlations = {
    monkey: rowwise_similarity(*tuning_responses[monkey], metric='spearman')
    for monkey in cfg.monkeys
}

figure, axis = plt.subplots(figsize=cfg.channel_bar_figsize)
bar_start = 0
tick_positions, tick_labels = [], []
for monkey in cfg.monkeys:
    correlations = channel_correlations[monkey]
    # One gap bar between monkeys keeps the groups visually separate.
    positions = bar_start + np.arange(correlations.size)
    axis.bar(
        positions, correlations, color=MONKEY_COLORS[monkey],
        edgecolor='white', label=monkey,
    )
    tick_positions.extend(positions)
    tick_labels.extend(monkey_data[monkey]['channel_numbers'])
    bar_start = positions[-1] + 2
# end for monkey
axis.axhline(0, color='0.25', linewidth=1)
axis.set_ylim(cfg.channel_correlation_ylim)
axis.set_xlim(-1, bar_start - 1)
style_axis(axis, 'Channel', r'Spearman $\rho$')
axis.set_xticks(tick_positions)
axis.set_xticklabels(tick_labels, rotation=90, fontsize=cfg.channel_tick_size)
axis.legend(frameon=False, fontsize=cfg.legend_size, ncol=len(cfg.monkeys))
figure.tight_layout()
save_figure(figure, 'all_monkeys_fig1d_channel_static_dynamic_correlation')
plt.show()
for monkey, correlations in channel_correlations.items():
    print(f'{monkey}: median rho = {np.nanmedian(correlations):.2f} over {correlations.size} channels')
# end for monkey

## Fig 1e — Peak-latency difference of the population response

For every stimulus the responses are averaged over channels, Gaussian-smoothed (`latency_smoothing_sigma_ms`), and the peak is searched in `latency_static_window_ms` (last frame, from image onset) and `latency_dynamic_window_ms` (movie, then expressed relative to the last-frame onset). The histogram shows the per-stimulus difference static − dynamic (positive = the static response peaks later); the box gives the mean and the paired t-test across stimuli.

In [ ]:
latency_results = {}
for monkey in cfg.monkeys:
    rasters = monkey_data[monkey]['rasters']
    latency_inputs = {
        # channels x time x stimuli, and its channel average (1 x time x stimuli).
        'channel': (rasters['last_frame'], rasters['dynamic']),
        'population': (
            rasters['last_frame'].mean(axis=0, keepdims=True),
            rasters['dynamic'].mean(axis=0, keepdims=True),
        ),
    }
    latency_results[monkey] = {}
    for level, (static_input, dynamic_input) in latency_inputs.items():
        _, static_peak_ms, _, _ = window_smoothed_response_latencies(
            static_input, cfg.latency_static_window_ms, cfg.source_fs,
            cfg.latency_smoothing_sigma_ms,
        )
        _, dynamic_peak_ms, _, _ = window_smoothed_response_latencies(
            dynamic_input, cfg.latency_dynamic_window_ms, cfg.source_fs,
            cfg.latency_smoothing_sigma_ms,
        )
        # Movie peaks relative to the last-frame onset; rows x stimuli.
        latency_results[monkey][level] = (
            static_peak_ms, dynamic_peak_ms - cfg.last_frame_onset_ms,
        )
    # end for level
# end for monkey

for monkey in cfg.monkeys:
    static_peak_ms, dynamic_peak_ms = latency_results[monkey]['population']
    latency_differences_ms = static_peak_ms[0] - dynamic_peak_ms[0]
    paired_test = ttest_rel(static_peak_ms[0], dynamic_peak_ms[0])

    figure, axis = plt.subplots(figsize=cfg.histogram_figsize)
    axis.hist(
        latency_differences_ms, bins=cfg.histogram_bin_count,
        color=cfg.dynamic_color, alpha=0.85, edgecolor='white',
    )
    axis.axvline(0, color='0.25', linewidth=1)
    axis.axvline(latency_differences_ms.mean(), color='black', linestyle='--', linewidth=1.5)
    axis.text(
        0.05, 0.95,
        f'mean = {latency_differences_ms.mean():.1f} ms\n'
        f't = {paired_test.statistic:.2f}, p = {paired_test.pvalue:.2g}',
        transform=axis.transAxes, ha='left', va='top',
        fontsize=cfg.latency_scatter_stats_fontsize,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor='black', alpha=0.9),
    )
    style_axis(axis, 'Static − dynamic peak latency (ms)', 'Number of stimuli', monkey)
    figure.tight_layout()
    save_figure(figure, f'{monkey}_fig1e_population_peak_latency_difference')
    plt.show()
# end for monkey

## Fig 1f — Peak-latency difference per channel (paired t score, pooled across monkeys)

Same latency windows and smoothing as Fig 1e, computed for every channel and stimulus. For every channel, the paired t score across stimuli compares the static peak latency with the movie peak latency relative to the last-frame onset (positive = the static response peaks later). Channels with identical latencies in both conditions have no t score and are left out.

In [ ]:
latency_t_scores = {}
for monkey in cfg.monkeys:
    static_peak_ms, dynamic_peak_ms = latency_results[monkey]['channel']
    with np.errstate(divide='ignore', invalid='ignore'):
        latency_t_scores[monkey] = ttest_rel(
            static_peak_ms, dynamic_peak_ms, axis=1,
        ).statistic
    # end with np.errstate
# end for monkey

figure, pooled_latency_t_scores = plot_pooled_channel_histogram(
    latency_t_scores, 'Paired t score (static − dynamic latency)',
)
save_figure(figure, 'all_monkeys_fig1f_peak_latency_difference_tscores')
plt.show()
print(
    f'{pooled_latency_t_scores.size} channels with a finite t | '
    f'mean t = {pooled_latency_t_scores.mean():.2f}'
)

## Static–dynamic dRSA (shared by Fig 2a, 2b and 2d)

Movie time × static time correlation (`rsa_metric`) of the `neural_rdm_metric` RDMs at `new_fs`, for the raw last-frame response and after regressing each response in `regressed_conditions` out of it (channel by channel and static bin, across stimuli, as in `static_dynamic_drsa_permutation.ipynb`).

In [ ]:
static_dynamic_drsa = {}
for monkey in cfg.monkeys:
    resampled = monkey_data[monkey]['resampled']
    # Dict of movie time x static time matrices: 'raw' plus every regressed frame.
    static_dynamic_drsa[monkey] = regressed_static_dynamic_drsa(
        resampled['last_frame'], resampled['dynamic'],
        {name: resampled[name] for name in cfg.regressed_conditions},
        rdm_metric=cfg.neural_rdm_metric, rsa_metric=cfg.rsa_metric,
    )
# end for monkey
drsa_bin_ms = 1000 / cfg.new_fs

## Fig 2a — Static–dynamic dRSA matrix with permutation significance

Loaded from the saved stimulus-permutation results (`results/static_dynamic_drsa_permutation`, condition `permutation_condition`). Cells significant under `significance_correction` (`'pointwise'`, `'max_statistic'` or `'cluster'`) at `significance_alpha` are outlined and kept sharp; the other cells are blurred (`nonsignificant_blur_sigma_bins`) and faded (`nonsignificant_fade_alpha`). The colour scale is fitted to each monkey unless `static_dynamic_matrix_vmin`/`vmax` are set.

In [ ]:
permutation_results = {}
for monkey in cfg.monkeys:
    sessions = MONKEY_SESSIONS[monkey]
    permutation_path = (
        PROJECT_ROOT / 'results' / 'static_dynamic_drsa_permutation'
        / f"{sessions['dynamic_exp_name']}_vs_{sessions['static_exp_name']}"
        / f'{cfg.neural_rdm_metric}_{cfg.rsa_metric}_{cfg.permutation_condition}_'
          f'{cfg.n_permutations}perm_seed{cfg.permutation_seed}.npz'
    )
    with np.load(permutation_path, allow_pickle=False) as archive:
        result = {key: archive[key] for key in archive.files}
    # end with np.load
    # The saved result must describe the channels and stimuli loaded above.
    if not np.array_equal(result['channel_numbers'], monkey_data[monkey]['channel_numbers']):
        raise ValueError(f'{monkey}: permutation result uses different channels.')
    # end if channels differ
    if result['shared_stimuli'].tolist() != monkey_data[monkey]['shared_stimuli']:
        raise ValueError(f'{monkey}: permutation result uses different stimuli.')
    # end if stimuli differ
    if cfg.permutation_condition == 'raw':
        live_difference = np.nanmax(np.abs(
            result['observed'] - static_dynamic_drsa[monkey]['raw']
        ))
        print(f'{monkey}: max |saved - recomputed| raw dRSA = {live_difference:.2g}')
    # end if raw condition
    significant = permutation_significance_mask(
        result, cfg.significance_correction, cfg.significance_alpha,
    )
    permutation_results[monkey] = {**result, 'significant': significant}

    figure, axis = plt.subplots(figsize=cfg.static_dynamic_matrix_figsize)
    plot_significance_masked_matrix(
        axis, result['observed'], significant,
        result['dynamic_times_ms'][1] - result['dynamic_times_ms'][0],
        nonsignificant_blur_sigma=cfg.nonsignificant_blur_sigma_bins,
        nonsignificant_fade_alpha=cfg.nonsignificant_fade_alpha,
        contour_color=cfg.significance_contour_color,
        contour_linewidth=cfg.significance_contour_width,
        cmap=cfg.static_dynamic_matrix_cmap,
        vmin=cfg.static_dynamic_matrix_vmin,
        vmax=cfg.static_dynamic_matrix_vmax,
        last_frame_onset_ms=cfg.last_frame_onset_ms,
        ticks_size=cfg.ticks_size, label_size=cfg.label_size,
    )
    if cfg.show_monkey_titles:
        axis.set_title(monkey, fontsize=cfg.title_size)
    # end if title shown
    figure.tight_layout()
    save_figure(
        figure,
        f'{monkey}_fig2a_static_dynamic_dRSA_{cfg.permutation_condition}_'
        f'{cfg.significance_correction}',
    )
    plt.show()
    print(
        f'{monkey}: {significant.sum()} / {significant.size} significant cells '
        f'({cfg.significance_correction}, alpha = {cfg.significance_alpha})'
    )
# end for monkey

## Fig 2b — Best static match per movie time vs split-half reliability

Orange: for every movie timebin, the maximum of the raw static–dynamic dRSA over static times in `split_half_static_reference_window_ms`. Black: split-half reliability of the movie RDM at every timebin (movie repetitions of every stimulus split in two random halves, RDM of one half correlated with the other, mean ± SD over `split_half_n_repeats` splits, Spearman–Brown corrected when `split_half_use_spearman_brown`). The reliability comes from the presentation-level raster `dynamic_raster_exp_name`, restricted to the loaded channels and to stimuli with at least two repetitions.

In [ ]:
split_half_results = {}
for monkey in cfg.monkeys:
    sessions = MONKEY_SESSIONS[monkey]
    loader_data = {
        'channel_numbers': monkey_data[monkey]['channel_numbers'],
        'stimuli': monkey_data[monkey]['shared_stimuli'],
        'neural_fs': cfg.new_fs,
    }
    # channels x time x presentations at new_fs, same channels and stimuli.
    presentations, identities = load_dynamic_presentations(
        data_dir / f"{sessions['dynamic_raster_exp_name']}_raster_vid.mat",
        loader_data, cfg.video_plot_end_ms, source_fs=cfg.source_fs, margin_ms=0,
    )
    repetition_counts = Counter(identities)
    reliability_stimuli = [
        stimulus for stimulus in monkey_data[monkey]['shared_stimuli']
        if repetition_counts[stimulus] >= 2
    ]
    # split repetitions x movie time RDM correlations between halves.
    reliability_splits = compute_split_half_reliability(
        presentations, identities, reliability_stimuli, 'rsa',
        np.random.default_rng(cfg.split_half_seed), cfg.split_half_n_repeats,
        rdm_metric=cfg.neural_rdm_metric, rsa_metric=cfg.rsa_metric,
    )
    if cfg.split_half_use_spearman_brown:
        correction_k = cfg.split_half_spearman_brown_k
        reliability_splits = (
            correction_k * reliability_splits
            / (1 + (correction_k - 1) * reliability_splits)
        )
    # end if Spearman-Brown correction

    raw_drsa = static_dynamic_drsa[monkey]['raw']
    static_time_ms = np.arange(raw_drsa.shape[1]) * drsa_bin_ms
    reference_start_ms, reference_end_ms = cfg.split_half_static_reference_window_ms
    static_reference = (static_time_ms >= reference_start_ms) & (static_time_ms < reference_end_ms)
    # Both curves share the movie time axis up to the shorter of the two.
    n_movie_bins = min(raw_drsa.shape[0], reliability_splits.shape[1])
    split_half_results[monkey] = {
        'movie_time_ms': np.arange(n_movie_bins) * drsa_bin_ms,
        'best_static_dynamic': np.nanmax(raw_drsa[:n_movie_bins, static_reference], axis=1),
        'reliability_mean': np.nanmean(reliability_splits[:, :n_movie_bins], axis=0),
        'reliability_std': np.nanstd(reliability_splits[:, :n_movie_bins], axis=0),
        'n_stimuli': len(reliability_stimuli),
    }
    print(
        f"{monkey}: {sessions['dynamic_raster_exp_name']} | {len(identities)} presentations | "
        f'{len(reliability_stimuli)} / {len(loader_data["stimuli"])} stimuli with >= 2 repetitions'
    )
# end for monkey

for monkey in cfg.monkeys:
    result = split_half_results[monkey]
    figure, axis = plt.subplots(figsize=cfg.split_half_figsize)
    axis.plot(
        result['movie_time_ms'], result['best_static_dynamic'],
        color=cfg.split_half_static_dynamic_color, linewidth=cfg.linewidth,
        label='static–dynamic dRSA corr',
    )
    axis.plot(
        result['movie_time_ms'], result['reliability_mean'],
        color=cfg.split_half_reliability_color, linewidth=cfg.linewidth,
        label='split-half RDM reliability',
    )
    axis.fill_between(
        result['movie_time_ms'],
        result['reliability_mean'] - result['reliability_std'],
        result['reliability_mean'] + result['reliability_std'],
        color=cfg.split_half_reliability_color, alpha=cfg.split_half_std_alpha,
        label='_nolegend_',
    )
    axis.axhline(0, color='0.6', linewidth=1)
    axis.axvline(cfg.last_frame_onset_ms, linestyle='--', color='red')
    style_axis(axis, 'Time from video onset (ms)', r'RSA corr ($\rho$)', monkey)
    axis.legend(fontsize=cfg.legend_size, loc='lower left')
    figure.tight_layout()
    save_figure(figure, f'{monkey}_fig2b_split_half_reliability_and_static_dynamic')
    plt.show()
# end for monkey

## Fig 2c — Static frame RDMs vs a sliding movie window

As in `timed_frame_window_rsa.ipynb`: every static condition (2000-ms frame, 2250-ms frame, last frame) is averaged in `window_rsa_static_window_ms`, centred across stimuli and turned into an RDM. A movie window of the same width slides over the movie in `window_rsa_step_ms` steps, and each static RDM is correlated with the movie RDM at every position. Dashed lines mark the frame onsets; the orange band is `window_rsa_movie_window_ms`.

In [ ]:
window_rsa_frames = (
    ('2000ms', 'static condition (-500 ms)', cfg.frame_500_color),
    ('2250ms', 'static condition (-250 ms)', cfg.frame_250_color),
    ('last_frame', 'static condition (last frame)', cfg.static_color),
)
window_width_ms = cfg.window_rsa_static_window_ms[1] - cfg.window_rsa_static_window_ms[0]
for monkey in cfg.monkeys:
    resampled = monkey_data[monkey]['resampled']
    # Same keys as load_frame_similarity_rasters, built from the loaded rasters.
    window_data = {
        'dynamic': resampled['dynamic'],
        **{f'static_{frame}': resampled[frame] for frame, _, _ in window_rsa_frames},
        'neural_fs': cfg.new_fs,
    }
    window_rsa = compute_timed_frame_window_rsa(
        window_data,
        static_window_ms=cfg.window_rsa_static_window_ms,
        movie_window_ms=cfg.window_rsa_movie_window_ms,
        frame_times_ms={frame: STATIC_ONSETS_MS[frame] for frame, _, _ in window_rsa_frames},
        rdm_metric=cfg.window_rsa_rdm_metric,
        similarity_metric=cfg.window_rsa_similarity_metric,
        n_bootstraps=0,
    )
    window_centers_ms, sliding_curves = sliding_movie_window_rsa(
        window_data, window_rsa['static_rdms'], window_width_ms, cfg.window_rsa_step_ms,
        rdm_metric=cfg.window_rsa_rdm_metric,
        similarity_metric=cfg.window_rsa_similarity_metric,
    )

    figure, axis = plt.subplots(figsize=cfg.window_rsa_figsize)
    for frame, label, color in window_rsa_frames:
        axis.plot(
            window_centers_ms, sliding_curves[frame], color=color,
            linewidth=cfg.linewidth, label=label,
        )
        axis.axvline(STATIC_ONSETS_MS[frame], color=color, linestyle='--', linewidth=1)
    # end for frame
    if cfg.window_rsa_show_target_window:
        axis.axvspan(*cfg.window_rsa_movie_window_ms, color='tab:orange', alpha=0.15)
    # end if target window shown
    axis.axhline(0, color='0.6', linewidth=1)
    style_axis(axis, 'Movie window center (ms)', r'RDM corr ($\rho$)', monkey)
    axis.legend(frameon=False, fontsize=cfg.legend_size, loc='upper left')
    figure.tight_layout()
    save_figure(figure, f'{monkey}_fig2c_sliding_movie_window_rsa')
    plt.show()
    print(
        f'{monkey}: static vs movie {cfg.window_rsa_movie_window_ms} ms r = '
        + ', '.join(
            f'{frame} {r_value:.2f}'
            for frame, r_value in zip(window_rsa['frame_names'], window_rsa['static_movie_r'])
        )
    )
# end for monkey

## Fig 2d — Best last-frame match per movie time, raw and after regressing out earlier frames

For every movie timebin, the maximum over static times in `split_half_static_reference_window_ms` of the static–dynamic dRSA, for the raw last-frame response (same curve as Fig 2b) and after regressing out the 2000-ms and 2250-ms frame responses.

In [ ]:
max_similarity_labels = {
    'raw': 'last frame',
    '2000ms': 'last frame, 2000 ms regressed out',
    '2250ms': 'last frame, 2250 ms regressed out',
}
for monkey in cfg.monkeys:
    figure, axis = plt.subplots(figsize=cfg.split_half_figsize)
    for condition_name, drsa_matrix in static_dynamic_drsa[monkey].items():
        static_time_ms = np.arange(drsa_matrix.shape[1]) * drsa_bin_ms
        reference_start_ms, reference_end_ms = cfg.split_half_static_reference_window_ms
        static_reference = (static_time_ms >= reference_start_ms) & (static_time_ms < reference_end_ms)
        # Best static match of every movie timebin.
        best_static_similarity = np.nanmax(drsa_matrix[:, static_reference], axis=1)
        axis.plot(
            np.arange(drsa_matrix.shape[0]) * drsa_bin_ms, best_static_similarity,
            color=cfg.max_similarity_colors[condition_name], linewidth=cfg.linewidth,
            label=max_similarity_labels[condition_name],
        )
    # end for condition_name
    axis.axhline(0, color='0.6', linewidth=1)
    axis.axvline(cfg.last_frame_onset_ms, linestyle='--', color='red')
    style_axis(axis, 'Time from video onset (ms)', r'RSA corr ($\rho$)', monkey)
    axis.legend(fontsize=cfg.legend_size, loc='upper left')
    figure.tight_layout()
    save_figure(figure, f'{monkey}_fig2d_max_similarity_regressed_frames')
    plt.show()
# end for monkey

## Model-layer RSA (shared by Fig 3a–3f)

For every layer of `model_name`, the model RDM of the last frame is correlated with the static neural RDM at every static time (static condition), and the model RDMs of the first frame, the last frame, and the last frame after regressing out the `previous_frame_ms` model RDM (across stimulus pairs) are correlated with the movie neural RDM at every movie time. Curves are smoothed with `smoothing`; centroid (positive mass above `centroid_similarity_cutoff`) and peak latencies are measured inside one window per condition, shared by peaks and centroids, and expressed relative to the frame onset. The last frame and its residual share `last_frame_latency_window_ms`.

In [ ]:
model_features_dir = cfg.model_features_dir or Path(paths['data_path']) / 'models'
layer_conditions = {
    'static': {
        'onset_ms': 0, 'window_ms': cfg.static_latency_window_ms,
        'label': 'static', 'xlabel': 'Time from image onset (ms)',
    },
    'first_frame': {
        'onset_ms': cfg.first_frame_ms, 'window_ms': cfg.first_frame_latency_window_ms,
        'label': 'first frame', 'xlabel': 'Time from video onset (ms)',
    },
    'last_frame': {
        'onset_ms': cfg.last_frame_onset_ms, 'window_ms': cfg.last_frame_latency_window_ms,
        'label': 'last frame', 'xlabel': 'Time from video onset (ms)',
    },
    'last_frame_residual': {
        'onset_ms': cfg.last_frame_onset_ms, 'window_ms': cfg.last_frame_latency_window_ms,
        'label': f'last frame, {cfg.previous_frame_ms:g} ms regressed out',
        'xlabel': 'Time from video onset (ms)',
    },
}

layer_results = {}
for monkey in cfg.monkeys:
    resampled = monkey_data[monkey]['resampled']
    # time x stimulus-pair neural RDMs.
    static_neural_rdms = compute_rdm_timeseries(resampled['last_frame'], cfg.neural_rdm_metric)
    dynamic_neural_rdms = compute_rdm_timeseries(resampled['dynamic'], cfg.neural_rdm_metric)
    feature_names = [Path(name).name for name in monkey_data[monkey]['aligned_names']['movie']]
    layer_names, layer_similarity, variance_removed = model_frame_similarity_by_layer(
        static_neural_rdms, dynamic_neural_rdms, feature_names, model_features_dir,
        cfg.model_name, cfg.model_dataset_name, cfg.model_pooling, cfg.new_fs,
        cfg.first_frame_ms, cfg.previous_frame_ms, cfg.last_frame_onset_ms,
        cfg.model_rdm_metric, cfg.rsa_metric, cfg.residual_fit_intercept,
    )
    layer_depths = np.linspace(0, 1, len(layer_names))

    timing = {}
    for condition_name, condition in layer_conditions.items():
        similarity = layer_similarity[condition_name]
        time_ms = np.arange(similarity.shape[1]) * 1000 / cfg.new_fs
        smoothed, centroid_latency_ms, peak_latency_ms, _ = summarize_layer_timing(
            similarity, time_ms, condition['onset_ms'],
            condition['window_ms'], condition['window_ms'], cfg.smoothing,
            cfg.centroid_similarity_cutoff, cfg.peak_similarity_cutoff,
        )
        timing[condition_name] = {
            'time_ms': time_ms, 'smoothed': smoothed,
            'centroid': centroid_latency_ms, 'peak': peak_latency_ms,
        }
    # end for condition_name
    layer_results[monkey] = {
        'layer_names': layer_names, 'layer_depths': layer_depths,
        'timing': timing, 'variance_removed': variance_removed,
    }
    print(
        f'{monkey}: {len(layer_names)} layers | {cfg.previous_frame_ms:g}-ms model RDM '
        f'explains {100 * variance_removed.mean():.0f}% of the last-frame RDM on average'
    )
# end for monkey

layer_cmap = truncate_colormap(
    plt.get_cmap(cfg.latency_scatter_cmap),
    cfg.latency_scatter_cmap_min, cfg.latency_scatter_cmap_max,
)


"""
plot_layer_timecourses
OC_presentation layer timecourses of one monkey and condition: one smoothed
RSA curve per layer, coloured by depth, with centroid ticks below zero.
Script-specific: it reads the notebook's cfg and layer_results.

INPUT:
    - monkey: str -> key of layer_results
    - condition_name: str -> key of layer_conditions

OUTPUT:
    - figure: matplotlib.figure.Figure -> timecourse figure
"""
def plot_layer_timecourses(monkey, condition_name):
    condition = layer_conditions[condition_name]
    timing = layer_results[monkey]['timing'][condition_name]
    layer_colors = layer_cmap(layer_results[monkey]['layer_depths'])
    figure, axis = plt.subplots(figsize=cfg.layer_timecourse_figsize)
    if cfg.show_latency_window:
        axis.axvspan(*condition['window_ms'], color='0.9', zorder=0)
    # end if latency window shown
    for layer_index, layer_curve in enumerate(timing['smoothed']):
        axis.plot(
            timing['time_ms'], layer_curve, color=layer_colors[layer_index],
            linewidth=cfg.linewidth,
        )
        centroid_latency_ms = timing['centroid'][layer_index]
        if np.isfinite(centroid_latency_ms):
            # Ticks sit at the absolute centroid time, below the zero line.
            axis.vlines(
                centroid_latency_ms + condition['onset_ms'],
                -cfg.latency_line_centroid_tick_height, 0,
                color=layer_colors[layer_index],
                linewidth=cfg.latency_line_centroid_tick_width, zorder=15,
            )
        # end if centroid defined
    # end for layer_index
    axis.axhline(0, color='grey', linewidth=0.8)
    axis.axvline(condition['onset_ms'], color='grey', linewidth=0.8, linestyle='--')
    if cfg.layer_timecourse_ylim is not None:
        axis.set_ylim(cfg.layer_timecourse_ylim)
    else:
        # Keep the centroid ticks visible below zero.
        axis.set_ylim(bottom=min(axis.get_ylim()[0], -1.2 * cfg.latency_line_centroid_tick_height))
    # end if fixed y limits
    x_limits_ms = cfg.layer_timecourse_xlims_ms.get(condition_name)
    axis.set_xlim(x_limits_ms or (timing['time_ms'][0], timing['time_ms'][-1]))
    style_axis(axis, condition['xlabel'], r'dRSA corr ($\rho$)', f"{monkey} | {condition['label']}")
    figure.tight_layout()
    return figure
# EOF


"""
plot_depth_latency_scatter
OC_presentation temporal-score scatterplot: latency of every layer against
its normalized depth, with the depth-latency Spearman rho and p value.
Script-specific: it reads the notebook's cfg and layer_results.

INPUT:
    - monkey: str -> key of layer_results
    - condition_name: str -> key of layer_conditions
    - latency_name: str -> "centroid" or "peak"

OUTPUT:
    - figure: matplotlib.figure.Figure -> scatter figure
"""
def plot_depth_latency_scatter(monkey, condition_name, latency_name):
    latency_values_ms = layer_results[monkey]['timing'][condition_name][latency_name]
    layer_depths = layer_results[monkey]['layer_depths']
    layer_colors = layer_cmap(layer_depths)
    valid_layers = np.isfinite(latency_values_ms)
    figure, axis = plt.subplots(figsize=cfg.latency_scatter_figsize)
    axis.scatter(
        latency_values_ms[valid_layers], layer_depths[valid_layers],
        c=layer_colors[valid_layers], s=cfg.latency_scatter_size,
        edgecolors=cfg.latency_scatter_edge_color,
        linewidths=cfg.latency_scatter_edge_width,
    )
    score = depth_latency_score(layer_depths, latency_values_ms)
    if np.isfinite(score['pvalue']):
        mantissa, exponent = f"{score['pvalue']:.1e}".split('e')
        stats_label = f"rho={score['rho']:.2f}\npval={mantissa} × 10$^{{{int(exponent)}}}$"
    else:
        stats_label = f"rho={score['rho']:.2f}\npval=nan"
    # end if finite p value
    axis.text(
        0.05, 0.95, stats_label, transform=axis.transAxes, ha='left', va='top',
        bbox=dict(facecolor='white', edgecolor='black'),
        fontsize=cfg.latency_scatter_stats_fontsize, zorder=20,
    )
    if valid_layers.any():
        # Pad the x range so edge points are not clipped.
        latency_min_ms = latency_values_ms[valid_layers].min()
        latency_max_ms = latency_values_ms[valid_layers].max()
        x_padding_ms = max(
            cfg.latency_scatter_min_x_padding_ms,
            cfg.latency_scatter_x_padding_fraction * (latency_max_ms - latency_min_ms),
        )
        axis.set_xlim(latency_min_ms - x_padding_ms, latency_max_ms + x_padding_ms)
    # end if any valid layer
    axis.axvline(0, color='0.6', linewidth=1, linestyle='--')
    axis.set_yticks(cfg.latency_scatter_y_ticks)
    axis.set_ylim(-0.05, 1.05)
    axis.locator_params(axis='x', nbins=cfg.latency_scatter_n_ticks)
    style_axis(
        axis, f'{latency_name.capitalize()} latency (ms)', 'Layer depth',
        f"{monkey} | {layer_conditions[condition_name]['label']}",
        hide_top_right=False,
    )
    figure.tight_layout()
    return figure
# EOF

## Fig 3a — Static condition: layer RSA timecourses

Last-frame model RDM of every layer vs the static neural RDM over static time. The grey band is the latency window; ticks below zero mark each layer's centroid. One plot per monkey.

In [ ]:
for monkey in cfg.monkeys:
    figure = plot_layer_timecourses(monkey, 'static')
    save_figure(figure, f'{monkey}_fig3a_static_layer_timecourses_{cfg.model_name}')
    plt.show()
# end for monkey

## Fig 3b — First-frame slice: layer RSA timecourses

First-frame model RDM of every layer vs the movie neural RDM over movie time. The grey band is the latency window; ticks below zero mark each layer's centroid. One plot per monkey.

In [ ]:
for monkey in cfg.monkeys:
    figure = plot_layer_timecourses(monkey, 'first_frame')
    save_figure(figure, f'{monkey}_fig3b_first_frame_layer_timecourses_{cfg.model_name}')
    plt.show()
# end for monkey

## Fig 3c — Last-frame slice: layer RSA timecourses

Last-frame model RDM of every layer vs the movie neural RDM over movie time. The grey band is the latency window; ticks below zero mark each layer's centroid. One plot per monkey.

In [ ]:
for monkey in cfg.monkeys:
    figure = plot_layer_timecourses(monkey, 'last_frame')
    save_figure(figure, f'{monkey}_fig3c_last_frame_layer_timecourses_{cfg.model_name}')
    plt.show()
# end for monkey

## Fig 3d — Last-frame slice after regressing out the 2250-ms frame

Residual of the last-frame model RDM after regressing out the same layer's `previous_frame_ms` RDM, vs the movie neural RDM. Latencies use `last_frame_latency_window_ms`, like Fig 3c. The grey band is the latency window; ticks below zero mark each layer's centroid. One plot per monkey.

In [ ]:
for monkey in cfg.monkeys:
    figure = plot_layer_timecourses(monkey, 'last_frame_residual')
    save_figure(figure, f'{monkey}_fig3d_last_frame_residual_layer_timecourses_{cfg.model_name}')
    plt.show()
# end for monkey

## Fig 3e — Temporal score from centroid latencies

Layer depth against centroid latency (relative to the frame onset) for every condition of Fig 3a–3d; the box gives the depth–latency Spearman ρ (temporal score) and its p value. One plot per monkey and condition.

In [ ]:
for monkey in cfg.monkeys:
    for condition_name in layer_conditions:
        figure = plot_depth_latency_scatter(monkey, condition_name, 'centroid')
        save_figure(
            figure,
            f'{monkey}_fig3e_{condition_name}_centroid_temporal_score_{cfg.model_name}',
        )
        plt.show()
    # end for condition_name
# end for monkey

## Fig 3f — Temporal score from peak latencies

Layer depth against peak latency (relative to the frame onset) for every condition of Fig 3a–3d; the box gives the depth–latency Spearman ρ (temporal score) and its p value. One plot per monkey and condition.

In [ ]:
for monkey in cfg.monkeys:
    for condition_name in layer_conditions:
        figure = plot_depth_latency_scatter(monkey, condition_name, 'peak')
        save_figure(
            figure,
            f'{monkey}_fig3f_{condition_name}_peak_temporal_score_{cfg.model_name}',
        )
        plt.show()
    # end for condition_name
# end for monkey